# Holdout PQ with test-time augmentation and a ResNet-18 + ResNet-34 ensemble (no retraining)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To evaluate another crop model in the place of the ResNet-34, change only `MODEL` in the settings cell and attach that model's training output; the configurations `r34` and `r34_tta` are then named after it (`r50`, `r50_tta`):

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

It loads two checkpoints that already exist, predicts each of the 142 holdout frames, and scores grouped-holdout pooled per-view PQ for six configurations:

| config | models | flip TTA |
| --- | --- | --- |
| `r34` | ResNet-34 crops | no |
| `r34_tta` | ResNet-34 crops | yes |
| `ens_50_50`, `ens_30_70` | ResNet-18 512 px (weight 0.5 / 0.3) + ResNet-34 crops (0.5 / 0.7) | no |
| `ens_50_50_tta`, `ens_30_70_tta` | the same two ensembles | yes |

Each is scored at threshold 0.45, `close_size` 15, `min_area` 400 and on a small grid around it (thresholds 0.40, 0.45, 0.50; `close_size` 11, 15, 21, 25; `min_area` 400). `close_size` 0 is added for one reason only: the `r34` row at threshold 0.5, `close_size` 0, `min_area` 400 has to reproduce the training notebook's PQ (0.3831 for `r34`), which checks the loading and the tiling. The result is `/kaggle/working/holdout_tta_ensemble.csv`, sorted by pooled PQ, best first. The notebook does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle.

**The ResNet-18 here is not a crop model.** The only ResNet-18 checkpoint in this project is the 12-epoch whole-frame model at 512 px (`train-unet-r18-12ep-2026-10-31`, holdout PQ 0.2886). Its probability map is predicted on the frame area-averaged to 512 px and resized back to 2048 px with bilinear interpolation, so that it can be averaged with the native-resolution ResNet-34.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.checkpoint.CROP_CHECKPOINTS`. [VERIFY] re-upload it after this PR is merged; an older version stops at the imports.
3. The output of the training notebook of `MODEL`: by default `train-unet-r34-crops-80ep`.
4. The output of the notebook `train-unet-r18-12ep-2026-10-31`.

**Settings**

- Accelerator: **GPU T4**, a single T4.
- Internet: **On**, for the `pip install` of `segmentation-models-pytorch==0.6.0`. The checkpoints hold the encoders, so no encoder weights are downloaded.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Install the segmentation library

`segmentation-models-pytorch` **0.6.0** (MIT), the version that trained both checkpoints. Kaggle already has PyTorch; this cell does not install it.

In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Internet has to be On. Kaggle already has PyTorch; this cell only "
        "installs segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 3. Settings and where the weights are

- `THRESHOLDS`, `CLOSE_SIZES`, `MIN_AREAS`: the grid. The sizes 11 and 21 stand where 10 and 20 were asked for: `filament.baseline.label_foreground` only takes `0` or an odd kernel size. `0` is the cross-check described above.
- `REFERENCE`: the setting the six configurations are compared at (threshold 0.45, `close_size` 15, `min_area` 400).
- `ENSEMBLE_WEIGHTS`: `(ResNet-18, MODEL)`. Weights are normalized to sum to 1.
- `MODEL` is a name of `filament.checkpoint.CROP_CHECKPOINTS`. `WEIGHTS_PATH` and `R18_WEIGHTS_PATH` come from `filament.checkpoint` (that entry, `R18_12EP`). The helpers that open a weights file, and re-zip a Kaggle-extracted archive, are the ones of `notebooks/infer_unet_crops.ipynb`. Both checkpoints are checked against their epochs, encoder and, once pinned, `tensor_sha256`; the notebook stops on a mismatch. `ALLOW_UNVERIFIED_WEIGHTS = True` turns that into a warning; leave it `False`.
- `TTA` here is the average of four views of each frame: as is, flipped left-right, flipped top-bottom and both, each prediction flipped back. The ResNet-34 was trained with these flips as augmentation; the ResNet-18 was trained without any, so its TTA is the less safe of the two.
- `TILE_OVERLAP` (128), `INFER_BATCH_SIZE` (8) and `USE_AMP` are the training notebook's values for the ResNet-34. The ResNet-18 runs in float32, as it was trained.

In [ ]:
import zipfile

from filament.checkpoint import R18_12EP, crop_checkpoint

# ---- Which model: a name of filament.checkpoint.CROP_CHECKPOINTS --------------
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# ---- Grid --------------------------------------------------------------------
THRESHOLDS = (0.40, 0.45, 0.50)
CLOSE_SIZES = (0, 11, 15, 21, 25)
MIN_AREAS = (400,)
# The setting the configurations are compared at.
REFERENCE = (0.45, 15, 400)
# What the MODEL row at (0.5, 0, 400) has to reproduce; None until recorded.
TRAINING_NOTEBOOK_PQ = EXPECTED.holdout_pq

# ---- Ensemble: (ResNet-18 weight, MODEL weight) ------------------------------
ENSEMBLE_WEIGHTS = ((0.5, 0.5), (0.3, 0.7))

# ---- Weights and the checkpoint guard ---------------------------------------
# Default from the checkpoint config. Another .pt path loads that file; None
# searches /kaggle/working and /kaggle/input (skipping competitions).
WEIGHTS_PATH = EXPECTED.weights_path
R18_WEIGHTS_PATH = R18_12EP.weights_path
ALLOW_UNVERIFIED_WEIGHTS = False

# ---- Inference ---------------------------------------------------------------
TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
USE_AMP = True
R18_SIDE = 512
SEED = 0

OUTPUT_PATH = Path("/kaggle/working/holdout_tta_ensemble.csv")

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
PREFERRED_NAME = Path(EXPECTED.weights_path).name


def _pt_under(root: Path) -> list[Path]:
    """Every ``*.pt`` file or directory under ``root``.

    A directory counts because Kaggle extracts an uploaded ``.pt`` zip
    into a folder of that name. ``competitions`` is not walked.
    """
    found: list[Path] = []
    if not root.is_dir():
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        directory = Path(dirpath)
        kept: list[str] = []
        for name in sorted(dirnames):
            if name == "competitions":
                continue
            if name.endswith(".pt"):
                found.append(directory / name)
                continue
            kept.append(name)
        dirnames[:] = kept
        for name in sorted(filenames):
            if name.endswith(".pt"):
                found.append(directory / name)
    return found


def _weight_candidates() -> list[Path]:
    """``.pt`` files and extracted archives, working then input."""
    return _pt_under(KAGGLE_WORKING) + _pt_under(KAGGLE_INPUT)


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )


def _resolve_weights(explicit):
    """Return the checkpoint file to load.

    ``explicit`` that exists is used (an extracted archive is re-zipped). The
    default path missing, or ``None``, searches ``/kaggle/working`` and
    ``/kaggle/input``: a file or folder named ``PREFERRED_NAME`` first, then
    a lone ``.pt``. A path that was set by hand and is missing is an error.
    """
    if explicit is not None:
        path = Path(explicit)
        if path.exists():
            return _materialize_checkpoint(path)
        if str(explicit) != EXPECTED.weights_path:
            raise FileNotFoundError(
                f"WEIGHTS_PATH is set to {path}, but that file does not exist."
            )
        print(f"{path} does not exist. Searching for {PREFERRED_NAME}.")
    candidates = _weight_candidates()
    named = [path for path in candidates if path.name == PREFERRED_NAME]
    if len(named) == 1:
        return _materialize_checkpoint(named[0])
    if not named and len(candidates) == 1:
        print(f"No {PREFERRED_NAME}; using the only .pt found: {candidates[0]}")
        return _materialize_checkpoint(candidates[0])
    listed = "\n".join(f"  {path}" for path in candidates) or "  (none)"
    detail = (
        f"Several files named {PREFERRED_NAME} were found."
        if named
        else f"No {PREFERRED_NAME} and not exactly one .pt were found."
    )
    raise RuntimeError(
        f"{detail} Attach the output of the notebook "
        f"{Path(EXPECTED.weights_path).parent.name}, or "
        f"set WEIGHTS_PATH to the file to load.\n{listed}"
    )

## 4. Load both checkpoints and check them

Prints the stored fields and `tensor_sha256` of each, runs the guard, and rebuilds the networks from the checkpoints with `encoder_weights=None`. The cell stops if there is no CUDA GPU.

In [ ]:
import random

import numpy as np
import torch

from filament import crop_training as ct
from filament.checkpoint import checkpoint_fingerprint, verify_and_report

if not hasattr(ct, "tiled_predictor"):
    raise RuntimeError(
        "The attached solar-code dataset predates the TTA / ensemble code "
        "(filament.crop_training.tiled_predictor). Upload a new version of "
        "the dataset from a checkout that has it."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = ct.require_cuda(6.0)
print(f"device={torch.cuda.get_device_name(0)} torch={torch.__version__}")


def _load_checked(path, expected, label):
    """Open a weights file, print its fingerprint, and run the guard."""
    weights_file = _resolve_weights(path)
    print(f"{label}: loaded weights={weights_file}")
    checkpoint = ct.load_checkpoint(weights_file)
    fingerprint = checkpoint_fingerprint(checkpoint)
    for key in ("epochs", "status", "encoder_name", "side", "crop_size", "smp_version"):
        print(f"  {key}={checkpoint.get(key, 'not stored')}")
    print(f"  tensor_sha256={fingerprint['tensor_sha256']}")
    verify_and_report(
        fingerprint,
        allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
        expected=expected,
    )
    return checkpoint


crop_checkpoint_data = _load_checked(WEIGHTS_PATH, EXPECTED, f"{MODEL} crops")
TILE_SIZE = int(
    crop_checkpoint_data.get("tile_size") or crop_checkpoint_data["crop_size"]
)
crop_model = ct.model_from_checkpoint(crop_checkpoint_data, device)
del crop_checkpoint_data

checkpoint18 = _load_checked(R18_WEIGHTS_PATH, R18_12EP, "r18 512 px")
model18 = ct.whole_frame_model_from_checkpoint(checkpoint18, device, side=R18_SIDE)
del checkpoint18

PREDICTORS = {
    MODEL: ct.tiled_predictor(
        crop_model,
        device,
        tile_size=TILE_SIZE,
        overlap=TILE_OVERLAP,
        batch_size=INFER_BATCH_SIZE,
        use_amp=USE_AMP,
    ),
    "r18": ct.resized_predictor(model18, device, side=R18_SIDE),
}
print(
    f"{MODEL}: tile={TILE_SIZE} overlap={TILE_OVERLAP} batch={INFER_BATCH_SIZE} amp={USE_AMP}"
)
print(f"r18: whole frame at {R18_SIDE} px, bilinear resize back, float32")

## 5. Date holdout

The same split as the training notebook: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Only the train images and the train JSON are read.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.sweep import count_views
from filament.train_data import split_training_images

IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
split = split_training_images(loaded.images)
holdout_views = count_views(split.holdout)
print(
    f"train files={len(split.train)} holdout files={len(split.holdout)} "
    f"holdout annotator views={holdout_views}"
)
if (len(split.train), len(split.holdout), holdout_views) != (565, 142, 231):
    print(
        "WARNING: the split is not 565 / 142 files with 231 views, the numbers "
        "the training notebook used. [VERIFY] the data version."
    )
print("test/test_images is not used")

## 6. Predict, combine, score

For each frame, each model predicts **once per flip that any configuration needs**: four flips of `MODEL` and four of the ResNet-18, kept in memory for that frame. Every configuration is then built from those maps (`filament.ensemble.combine`): the mean of the flips of each model, then the weighted mean across models. A configuration costs no extra network pass. Each map is scored on the grid by `filament.sweep.sweep_postproc_configs`, with the pooled per-view PQ of `filament.sweep` (`fast_view_counts`, `PqCounts`, one `.pq` at the end).

Eight network passes per frame (four of them tiled at 25 tiles) make inference about four times as long as the training notebook's holdout evaluation. The total duration was not measured [VERIFY]; the progress line prints every 10 frames.

One holdout is one sample of dates, and 6 configurations × 15 settings are compared on it. A gain of a few thousandths of PQ is within what that selection can produce on its own. Prefer a configuration that is better across the grid, not only at its best cell.

In [ ]:
from filament.crop_data import read_gray
from filament.ensemble import EvalConfig
from filament.sweep import (
    config_rows,
    count_views,
    format_config_table,
    sweep_configs,
    write_config_rows,
)

CONFIGS = [
    EvalConfig(MODEL, {MODEL: 1.0}),
    EvalConfig(f"{MODEL}_tta", {MODEL: 1.0}, tta=True),
]
for w18, w_model in ENSEMBLE_WEIGHTS:
    tag = f"ens_{round(w18 * 100)}_{round(w_model * 100)}"
    CONFIGS.append(EvalConfig(tag, {"r18": w18, MODEL: w_model}))
    CONFIGS.append(EvalConfig(f"{tag}_tta", {"r18": w18, MODEL: w_model}, tta=True))
print("configs:", [config.name for config in CONFIGS])


def _read(image):
    return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))


def _progress(index, total, image):
    if index == 1 or index % 10 == 0 or index == total:
        print(f"  holdout {index}/{total} {image.file_name}", flush=True)


sweep = sweep_configs(
    split.holdout,
    _read,
    PREDICTORS,
    CONFIGS,
    THRESHOLDS,
    CLOSE_SIZES,
    MIN_AREAS,
    progress=_progress,
)
rows = config_rows(
    CONFIGS, sweep.totals, holdout_files=len(split.holdout), views=holdout_views
)
write_config_rows(OUTPUT_PATH, rows)
print(format_config_table(rows, limit=15))
print(f"wrote {len(rows)} rows to {OUTPUT_PATH} (top 15 shown)")

## 7. Compare the configurations at the reference setting

The six configurations at threshold 0.45, `close_size` 15, `min_area` 400, best first, and the cross-check: `MODEL` at threshold 0.5, `close_size` 0, `min_area` 400 has to match the training notebook's PQ (the checkpoint entry's `holdout_pq`, 0.3831 for `r34`) to within about 0.005. With no recorded PQ there is no cross-check. If it does not, the weights, the data or the tiling are not the same as in that run, and none of the other rows should be trusted until that is explained.

In [ ]:
cut, kernel, area = REFERENCE


def _at(setting, name=None):
    t, k, a = setting
    return [
        row
        for row in rows
        if (float(row["threshold"]), int(row["close_size"]), int(row["min_area"]))
        == (t, k, a)
        and (name is None or row["config"] == name)
    ]


print(f"pooled PQ at threshold {cut}, close_size {kernel}, min_area {area}:")
reference_rows = _at(REFERENCE)
for row in reference_rows:
    print(
        f"  {row['config']:<16} {row['models']:<18} tta={row['tta']:<3} {row['pooled_pq']}"
    )

check = _at((0.5, 0, 400), MODEL)
if check and TRAINING_NOTEBOOK_PQ is None:
    print(f"No holdout_pq is recorded for {MODEL!r}, so there is no cross-check.")
elif check:
    value = float(check[0]["pooled_pq"])
    print(
        f"cross-check {MODEL} at (0.5, 0, 400): {value:.4f} vs {TRAINING_NOTEBOOK_PQ}"
    )
    if abs(value - TRAINING_NOTEBOOK_PQ) > 0.005:
        print("WARNING: the cross-check differs from the training notebook's PQ.")
print(f"best row overall: {rows[0]}")
print(
    f"inference {sweep.inference_seconds:.0f} s, "
    f"inference + labeling + scoring {sweep.total_seconds:.0f} s"
)
print("Output: holdout_tta_ensemble.csv in /kaggle/working.")
print("This notebook wrote no submission file and did not submit to Kaggle.")